> ## SOLUTION / ANSWER KEY &mdash; Lab 7.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-retry-idempotency.ipynb`](../lab-08-retry-idempotency.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.8 &mdash; Retry & Idempotency

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Retry a flaky order lookup, but cap the attempts so it can't loop forever
- Make sending idempotent with an order+draft key, so a re-run is safe
- See why idempotency matters most for money & messages

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Reliability: retry & idempotency](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-08"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Models and tools are flaky, so wrap calls in a **retry** &mdash; but **cap** the attempts (deck slide
12). And design for **idempotency**: key each action so running the same task twice is **safe** and
never **double-sends** an email or **double-charges** a card. This is the subtlest and most important
discipline for anything that touches **money or messages**. (Note: the `with_backoff` in your Setup
cell is exactly this pattern applied to Groq's rate limits.)

In [ ]:
# A flaky order lookup: the network hiccups n_fail times, then returns the order.
import hashlib
def flaky_lookup(order_id, n_fail):
    calls = {"n": 0}
    def f():
        calls["n"] += 1
        if calls["n"] <= n_fail:
            raise RuntimeError("transient network error")
        return {"id": order_id, "status": "shipped", "eta": "Friday"}
    return f
def raises(fn):
    try: fn(); return False
    except Exception: return True
print("helpers ready: flaky_lookup(id, n) fails n times, then returns the order dict")

## Build it
Complete `with_retry` (capped), `send_key` (the idempotency key from the order id + a draft hash),
and `send_once` (idempotent via the key set).

In [ ]:
import hashlib

def with_retry(fn, max_attempts=3):
    # call fn(); on failure retry up to max_attempts; raise the last error if all fail
    last = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last = e
    raise last

def send_key(order_id, draft):
    # the idempotency key ties the EXACT draft to its order -- re-sending the same one is a no-op
    h = hashlib.sha256(draft.encode()).hexdigest()[:8]
    return f"{order_id}:{h}"

def send_once(key, sent):
    # idempotent: sending the same key twice must NOT double-send
    if key in sent:
        return "already_sent"
    sent.add(key)
    return "sent"

try:
    print("first try ok   :", with_retry(flaky_lookup("4471", 0))["status"])
    print("after 2 fails  :", with_retry(flaky_lookup("4471", 2), 3)["status"])
    print("exhausted raises:", raises(lambda: with_retry(flaky_lookup("4471", 5), 3)))
    k = send_key("4471", "Hi Priya, your order 4471 is due Friday.")
    print("send key        :", k)
    sent = set()
    print("send (1st)      :", send_once(k, sent))
    print("send (2nd)      :", send_once(k, sent))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `with_retry` succeeds *after* transient failures but **raises once the cap is hit** &mdash; bounded, never an infinite loop.
- `send_key` ties the **exact draft** to its order; a different draft yields a different key, so an edited reply is a new send.
- `send_once` makes the second identical send a **no-op** &mdash; the property that lets an automation re-run safely.

## Your turn (open task &mdash; no grader)
Change `send_key` to ignore the draft (key on `order_id` alone) and watch what breaks: a legitimately
*revised* reply to the same order would now be suppressed. **What good looks like:** you can articulate why
the key must include the draft hash &mdash; idempotency should block *duplicates*, not *revisions*.

---
### Key takeaway
Retry with a cap, and key every irreversible action so a re-run is safe. Assume every step can fail -- and make failure safe and visible. Idempotency is what lets an automation run unattended.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>